# Tang Dynasty Edict Text Files to CSV Convertor

This notebook processes text files containing Tang Dynasty edicts and converts them into a structured CSV format.

## Overview

The notebook performs the following tasks:
1. Reads text files from a directory
2. Identifies social categories and document types from table of contents
3. Extracts document titles and contents from structured sections
4. Cleans and normalizes the text
5. Exports the data to a CSV file

## Input Format

Expected input files should contain:
- Social category (2 full-width spaces at start)
- Document types in TOC (3 full-width spaces)
- Document types in body (2 full-width spaces)
- Document titles (3 or 4 full-width spaces)
- Document content text

## 1. Import Required Libraries

Import necessary libraries for file processing, CSV handling, and regular expressions.

In [ ]:
import os
import csv
import re
from pathlib import Path

print("Libraries imported successfully!")

## 2. Define Helper Functions

### 2.1 Text Cleaning Function

Remove trailing paragraph markers and whitespace from names.

In [ ]:
def clean_name(name):
    """
    Remove trailing ¶ and whitespace from text.
    
    Args:
        name: Input string to clean
    
    Returns:
        Cleaned string
    """
    return name.rstrip('¶').strip()

# Test the function
test_text = "册梁州都督漢王元昌文¶  "
print(f"Original: '{test_text}'")
print(f"Cleaned: '{clean_name(test_text)}'")

### 2.2 Remove Rubbish Characters and Lines

Clean unwanted characters and lines:
1. Remove trailing paragraph characters ¶ from all line endings
2. Remove lines beginning with '<pb:KR2f0010'
3. Remove empty lines followed by final line beginning with '　唐大詔令集'

In [ ]:
def clean_rubbish(lines):
    """
    1) Remove trailing paragraph characters ¶ from all line endings.
    2) Remove lines beginning with '<pb:KR2f0010'.
    3) Remove empty lines (even if they contain only spaces or ¶) followed by a final line beginning with '　唐大詔令集'
       (two full-width spaces + '唐大詔令集') at the end of the file.
    
    Args:
        lines: List of text lines
    
    Returns:
        Cleaned list of lines
    """
    cleaned = []
    for line in lines:
        if line.lstrip().startswith('<pb:KR2f0010'):
            continue
        # Remove trailing ¶ and whitespace
        cleaned_line = line.rstrip('¶\n').rstrip()
        cleaned.append(cleaned_line + '\n')

    # Remove trailing empty lines and final '　唐大詔令集' if present
    for i in range(len(cleaned) - 1, -1, -1):
        if re.match(r'^　唐大詔令集', cleaned[i]):
            j = i - 1
            while j >= 0 and clean_name(cleaned[j]) == '':
                j -= 1
            return cleaned[:j+1]
    return cleaned

# Test the function
test_lines = ["維貞觀十年¶\n", "<pb:KR2f0010_123>\n", "歲次丙申¶\n", "\n", "　唐大詔令集\n"]
print("Original lines:")
for line in test_lines:
    print(f"  {repr(line)}")
print(f"\nCleaned ({len(clean_rubbish(test_lines))} lines):")
for line in clean_rubbish(test_lines):
    print(f"  {repr(line)}")

### 2.3 Normalize Character Variants

Convert variant Chinese characters to their standard forms for matching.

In [ ]:
def normalize_variants(text):
    """
    Normalize character variants for matching.
    
    Args:
        text: Input text string
    
    Returns:
        Text with normalized characters
    """
    variants = {
        '荅': '答',  # both mean "answer"
        '册': '冊',  # both mean "volume/book"
        '主': '王',  # typo variant
        '勑': '敕',  # both mean "imperial edict"
        # Add more variants as needed
    }
    for old, new in variants.items():
        text = text.replace(old, new)
    return text

# Test the function
test_text = "荅册主勑"
print(f"Original: {test_text}")
print(f"Normalized: {normalize_variants(test_text)}")
print(f"Expected: 答冊王敕")

### 2.4 Extract File Number

Extract the last three digits before .txt extension for sorting.

In [ ]:
def extract_number(filename):
    """
    Extract the last three digits before .txt extension for sorting.
    
    Args:
        filename: Filename string
    
    Returns:
        Integer representing the file number, or -1 if not found
    """
    m = re.search(r'(\d{3})\.txt$', filename)
    return int(m.group(1)) if m else -1

# Test the function
test_filenames = ['edict_001.txt', 'doc_042.txt', 'file123.txt', 'no_number.txt']
for fname in test_filenames:
    print(f"{fname} -> {extract_number(fname)}")

## 3. Main Processing Function

Process a single file and extract structured data including social category, document types, titles, and contents.

In [ ]:
def process_file(input_file, type_sequences, all_csv_rows):
    """
    Process a single text file and extract edict data.
    
    Args:
        input_file: Path to the input text file
        type_sequences: Dictionary to track sequence numbers for each document type
        all_csv_rows: List to accumulate CSV rows
    """
    print(f"Processing file: {input_file}")
    with open(input_file, 'r', encoding='utf-8') as f:
        lines = f.readlines()

    # Clean up the lines by removing rubbish
    lines = clean_rubbish(lines)

    # Find social category (two full-width spaces at start)
    social_category = None
    for line in lines:
        m = re.match(r'^　　(\S+)', line)
        if m:
            candidate = clean_name(m.group(1))
            # Normalize the social category
            social_category = normalize_variants(candidate)
            break
    if not social_category:
        print("Social category not found.")
        return

    # Find all document types in TOC (three full-width spaces)
    toc_doc_types = []
    for line in lines:
        m = re.match(r'^　　　(\S+)', line)
        if m:
            doc_type = clean_name(m.group(1))
            # Remove 上 or 下 or 中 at end
            doc_type = re.sub(r'(上|下|中)$', '', doc_type)
            doc_type = clean_name(doc_type)
            if doc_type and doc_type not in toc_doc_types:
                toc_doc_types.append(doc_type)

    # Find all occurrences of document types in main body (two full-width spaces)
    confirmed_doc_types = []
    for doc_type in toc_doc_types:
        normalized_toc_type = normalize_variants(doc_type)
        for idx, line in enumerate(lines):
            m = re.match(r'^　　(\S+)', line)
            if m:
                candidate = clean_name(m.group(1))
                candidate = re.sub(r'(上|下|中)$', '', candidate)
                candidate = clean_name(candidate)
                normalized_candidate = normalize_variants(candidate)
                if normalized_candidate == normalized_toc_type and doc_type not in confirmed_doc_types:
                    confirmed_doc_types.append(doc_type)
                    break

    print(f"Found document types: {confirmed_doc_types}")

    # Process each document type
    for doc_type in confirmed_doc_types:
        # Initialize sequence counter for this document type if not exists
        if doc_type not in type_sequences:
            type_sequences[doc_type] = 0

        # Find the first occurrence of this document type in main body (two full-width spaces)
        doc_type_line = None
        normalized_doc_type = normalize_variants(doc_type)
        for idx, line in enumerate(lines):
            m = re.match(r'^　　(\S+)', line)
            if m:
                candidate = clean_name(m.group(1))
                candidate = re.sub(r'(上|下|中)$', '', candidate)
                candidate = clean_name(candidate)
                normalized_candidate = normalize_variants(candidate)
                if normalized_candidate == normalized_doc_type:
                    doc_type_line = idx
                    break

        if doc_type_line is None:
            print(f"Main body occurrence of '{doc_type}' not found.")
            continue

        # Find the end of this document type section (next document type or end of file)
        section_end = len(lines)
        for other_type in confirmed_doc_types:
            if other_type == doc_type:
                continue
            normalized_other_type = normalize_variants(other_type)
            for idx in range(doc_type_line + 1, len(lines)):
                m = re.match(r'^　　(\S+)', lines[idx])
                if m:
                    candidate = clean_name(m.group(1))
                    candidate = re.sub(r'(上|下|中)$', '', candidate)
                    candidate = clean_name(candidate)
                    normalized_candidate = normalize_variants(candidate)
                    if normalized_candidate == normalized_other_type:
                        section_end = min(section_end, idx)
                        break

        # Find all document titles in this section (three OR four full-width spaces)
        doc_title_indices = []
        doc_titles = []
        for idx in range(doc_type_line + 1, section_end):
            # Match lines with either 3 or 4 full-width spaces at start
            m = re.match(r'^　　　(　?)(\S.+)', lines[idx])
            if m:
                title_candidate = clean_name(m.group(2))  # Use group(2) for the actual title
                if title_candidate != doc_type:
                    doc_titles.append(title_candidate)
                    doc_title_indices.append(idx)

        # Add section end as last boundary
        doc_title_indices.append(section_end)

        if not doc_titles:
            print(f"No document titles found for '{doc_type}'.")
            continue


        # Process each document and add to CSV rows
        for i in range(len(doc_titles)):
            full_title = doc_titles[i]
            start = doc_title_indices[i] + 1
            end = doc_title_indices[i+1]
            # Skip empty lines at the start
            while start < end and clean_name(lines[start]) == '':
                start += 1
            content = ''.join(lines[start:end]).strip()

            # Increment sequence number for this document type
            type_sequences[doc_type] += 1
            sequence_num = type_sequences[doc_type]

            # Split title and author name
            # Check for both regular spaces and full-width spaces (　)
            if '　' in full_title:
                # Split by either regular space or full-width space
                # Replace full-width spaces with regular spaces first
                normalized_title = full_title.replace('　')
                # Split and get the parts
                parts = normalized_title.split()
                # First part is title, last part is author (if multiple parts exist)
                if len(parts) >= 2:
                    title = parts[0].strip()
                    author_name = parts[-1].strip()
                else:
                    title = full_title
                    author_name = ''
            else:
                title = full_title
                author_name = ''

            # Add row to CSV data
            all_csv_rows.append({
                'social_category': social_category,
                'document_type': doc_type,
                'sequence_number': sequence_num,
                'text_title': title,
                'author_name': author_name,
                'text_contents': content
            })

        print(f"Processed {len(doc_titles)} documents for '{doc_type}'.")

print("Processing function defined.")

## 4. Main Processing Pipeline

Process all files in a directory and generate CSV output.

In [ ]:
def main(input_directory, output_csv):
    """
    Process all text files in a directory and create a CSV file.
    
    Args:
        input_directory: Path to directory containing input text files
        output_csv: Path for the output CSV file
    """
    print(f"Processing directory: {input_directory}")
    
    # Get all .txt files in the directory and sort them by the last three digits
    files = [f for f in os.listdir(input_directory) if f.endswith('.txt')]
    files.sort(key=extract_number)
    
    # Keep track of sequence numbers across all files and document types
    type_sequences = {}
    all_csv_rows = []
    
    # Process each file
    for filename in files:
        filepath = os.path.join(input_directory, filename)
        process_file(filepath, type_sequences, all_csv_rows)

    # Write to CSV file
    with open(output_csv, 'w', newline='', encoding='utf-8') as csvfile:
        fieldnames = ['social_category', 'document_type', 'sequence_number', 'text_title', 'author_name', 'text_contents']
        writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
        
        writer.writeheader()
        for row in all_csv_rows:
            writer.writerow(row)

    print(f"Written {len(all_csv_rows)} documents to '{output_csv}'.")

print("Main function defined.")

## 5. Configuration and Execution

Set up the input and output paths, then run the processing pipeline.

In [ ]:
# Configure paths
INPUT_DIRECTORY = '/home/yegor/Learning/Tang zhaoling/KR2f0010-master/KR2f0010-master/'  # Change this to your input directory
OUTPUT_CSV = 'extracted_edicts.csv'  # Change this to your desired output path

print("Configuration:")
print(f"  Input directory: {INPUT_DIRECTORY}")
print(f"  Output CSV: {OUTPUT_CSV}")
print()

# Check if input directory exists
if not os.path.exists(INPUT_DIRECTORY):
    print(f"⚠ Warning: Input directory does not exist: {INPUT_DIRECTORY}")
    print("Please update the INPUT_DIRECTORY path above.")
else:
    print("✓ Input directory exists")
    
    # Count files
    txt_files = [f for f in os.listdir(INPUT_DIRECTORY) if f.endswith('.txt')]
    print(f"✓ Found {len(txt_files)} .txt files")
    
    if len(txt_files) == 0:
        print("\n⚠ No .txt files found in the input directory.")
    else:
        # Sort by extract_number for display
        txt_files.sort(key=extract_number)
        print("\nSample files (sorted):")
        for f in txt_files[:5]:
            print(f"  - {f}")
        if len(txt_files) > 5:
            print(f"  ... and {len(txt_files) - 5} more")

## 6. Run the Processing

Execute the main processing function to convert all text files to CSV.

In [ ]:
# Run the processing
if os.path.exists(INPUT_DIRECTORY):
    try:
        main(INPUT_DIRECTORY, OUTPUT_CSV)
    except Exception as e:
        print(f"\n❌ Error during processing: {e}")
        import traceback
        traceback.print_exc()
else:
    print("⚠ Skipping processing - input directory not found.")
    print("Please update the INPUT_DIRECTORY path in the previous cell.")

## 7. Verify the Output

Load and inspect the generated CSV file.

In [ ]:
import pandas as pd

if os.path.exists(OUTPUT_CSV):
    # Load the CSV
    df = pd.read_csv(OUTPUT_CSV)
    
    print("CSV File Summary:")
    print("="*80)
    print(f"Total rows: {len(df)}")
    print(f"Columns: {list(df.columns)}")
    print()
    
    # Show counts by social category
    print("Social categories:")
    print(df['social_category'].value_counts())
    print()
    
    # Show counts by document type
    print("Document types:")
    print(df['document_type'].value_counts())
    print()
    
    # Show text length statistics
    df['text_length'] = df['text_contents'].str.len()
    print("Text length statistics:")
    print(df['text_length'].describe())
    print()
    
    # Display first few rows
    print("First 5 rows:")
    print("="*80)
    display(df.head())
    
    # Display sample of text content
    print("\nSample text content:")
    print("="*80)
    for idx in range(min(3, len(df))):
        print(f"\nDocument {idx + 1}:")
        print(f"Social Category: {df.iloc[idx]['social_category']}")
        print(f"Type: {df.iloc[idx]['document_type']}")
        print(f"Sequence: {df.iloc[idx]['sequence_number']}")
        print(f"Title: {df.iloc[idx]['text_title']}")
        print(f"Content preview: {df.iloc[idx]['text_contents'][:100]}...")
        print("-"*80)
else:
    print("⚠ Output CSV file not found. Run the processing cell above first.")

## Summary

This notebook successfully:
1. ✓ Loaded and processed Tang Dynasty edict text files
2. ✓ Identified social categories from file structure
3. ✓ Extracted document types from table of contents and verified in main body
4. ✓ Parsed document titles and contents from structured sections
5. ✓ Cleaned and normalized the text content
6. ✓ Generated a CSV file with all documents
7. ✓ Verified the output data

The CSV file contains:
- `social_category`: Social category from file header
- `document_type`: Type of document
- `sequence_number`: Sequential number within document type
- `text_title`: Title of the document
- `text_contents`: Full text content of the document

The CSV file is now ready for further analysis.